# 01 - Exploratory Data Analysis: Tabular Datasets

This notebook performs exploratory data analysis on six tabular datasets
from the Interior Minister (Ministerio del Interior) of Uruguay.

**Datasets covered:**
- `delitos_denunciados` - Reported crimes
- `violencia_domestica` - Domestic violence incidents
- `delitos_sexuales` - Sexual offenses
- `homicidios_mujeres` - Homicides of women
- `medidas_alternativas` - Alternative measures
- `sistema_carcelario` - Prison system

**Tools:** Polars for data wrangling, DuckDB for cross-dataset queries, Plotly for visualization.

In [1]:
!pip install -q polars pyarrow duckdb plotly

In [2]:
import polars as pl
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(80)

polars.config.Config

In [3]:
# --- Download from Google Cloud Storage ---
from google.colab import auth
auth.authenticate_user()

GCS_BUCKET = "gs://interior-minister-data-lake-fabled-imagery-488015-p6/processed/tabular"
DATA_DIR = Path("/content/data/tabular")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TABULAR_FILES = [
    "delitos_denuncias.parquet",
    "delitos_homicidios.parquet",
    "delitos_sexuales.parquet",
    "violencia_domestica.parquet",
    "homicidios_mujeres.parquet",
    "medidas_alternativas.parquet",
    "sistema_carcelario.parquet",
]

for f in TABULAR_FILES:
    !gsutil cp {GCS_BUCKET}/{f} {DATA_DIR}/{f}

print("\nDownloaded files:")
for p in sorted(DATA_DIR.glob("*.parquet")):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://interior-minister-data-lake-fabled-imagery-488015-p6/processed/tabular/delitos_denuncias.parquet...
- [1 files][ 18.1 MiB/ 18.1 MiB]                                                
Operation completed over 1 objects/18.1 MiB.                                     
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://interior-minister-data-lake-fabled-imagery-488015-p6/processed/tabular/delitos_homicidios.parquet...
/ [1 files][ 69.6 KiB/ 69.6 KiB]                                                
Operation c

In [4]:
# --- Load delitos_denuncias (2.4M reported crimes) ---
df_delitos = pl.read_parquet(DATA_DIR / "delitos_denuncias.parquet")

print(f"Shape: {df_delitos.shape}")
print(f"\nSchema:")
for col_name, dtype in df_delitos.schema.items():
    print(f"  {col_name}: {dtype}")

# Separate numeric vs string columns for meaningful stats
numeric_cols = [c for c, d in df_delitos.schema.items() if d in (pl.Int64, pl.Float64, pl.Int32, pl.Float32)]
string_cols = [c for c, d in df_delitos.schema.items() if d == pl.Utf8]
date_cols = [c for c, d in df_delitos.schema.items() if d == pl.Date]

print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"String columns ({len(string_cols)}): {string_cols}")
print(f"Date columns ({len(date_cols)}): {date_cols}")

# Numeric stats (only meaningful for numeric columns)
if numeric_cols:
    print(f"\nNumeric statistics:")
    print(df_delitos.select(numeric_cols).describe())

# String columns: show unique value counts
print(f"\nUnique values per string column:")
for col in string_cols:
    n_unique = df_delitos[col].n_unique()
    print(f"  {col}: {n_unique:,} unique values")

# Null counts
print(f"\nNull counts:")
null_counts = {c: df_delitos[c].null_count() for c in df_delitos.columns if df_delitos[c].null_count() > 0}
if null_counts:
    for c, n in null_counts.items():
        print(f"  {c}: {n:,} nulls ({n/len(df_delitos)*100:.1f}%)")
else:
    print("  No nulls in any column")

Shape: (2423286, 15)

Schema:
  id_evento: String
  delito: String
  vict_rap: String
  vict_hur: String
  tentativa: String
  fecha: Date
  ano: Int64
  mes: String
  semestre: String
  trimestre: String
  dia_semana: String
  hora: String
  depto: String
  jurisdiccion: String
  barrio_montevideo: String

Numeric columns (1): ['ano']
String columns (13): ['id_evento', 'delito', 'vict_rap', 'vict_hur', 'tentativa', 'mes', 'semestre', 'trimestre', 'dia_semana', 'hora', 'depto', 'jurisdiccion', 'barrio_montevideo']
Date columns (1): ['fecha']

Numeric statistics:
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ ano         │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 2.423286e6  │
│ null_count ┆ 0.0         │
│ mean       ┆ 2019.092541 │
│ std        ┆ 3.601822    │
│ min        ┆ 2013.0      │
│ 25%        ┆ 2016.0      │
│ 50%        ┆ 2019.0      │
│ 75%        ┆ 2022.0      │
│ max        ┆ 2025.0      │
└──────────

In [5]:
# --- Discover actual column names ---
print("Columns in delitos_denuncias:", df_delitos.columns)
print("\nFirst 5 rows:")
df_delitos.head(5)

Columns in delitos_denuncias: ['id_evento', 'delito', 'vict_rap', 'vict_hur', 'tentativa', 'fecha', 'ano', 'mes', 'semestre', 'trimestre', 'dia_semana', 'hora', 'depto', 'jurisdiccion', 'barrio_montevideo']

First 5 rows:


id_evento,delito,vict_rap,vict_hur,tentativa,fecha,ano,mes,semestre,trimestre,dia_semana,hora,depto,jurisdiccion,barrio_montevideo
str,str,str,str,str,date,i64,str,str,str,str,str,str,str,str
"""ABI161166""","""VIOLENCIA DOMÉSTICA""","""NO CORRESPONDE""","""NO CORRESPONDE""","""NO""",2013-01-01,2013,"""ENERO""","""PRIMER SEMESTRE""","""ENE-MAR""","""MARTES""","""0""","""MONTEVIDEO""","""SECCIONAL 10""","""POCITOS"""
"""BAF181532""","""VIOLENCIA DOMÉSTICA""","""NO CORRESPONDE""","""NO CORRESPONDE""","""NO""",2013-01-14,2013,"""ENERO""","""PRIMER SEMESTRE""","""ENE-MAR""","""LUNES""","""8""","""MONTEVIDEO""","""SECCIONAL 16""","""ITUZAINGO"""
"""BBB27344""","""HURTO""","""NO CORRESPONDE""","""DE VEHICULO""","""NO""",2013-01-01,2013,"""ENERO""","""PRIMER SEMESTRE""","""ENE-MAR""","""MARTES""","""SIN DATO""","""MALDONADO""","""SECCIONAL 1""","""NO CORRESPONDE"""
"""BBP141672""","""LESIONES""","""NO CORRESPONDE""","""NO CORRESPONDE""","""SI""",2013-01-16,2013,"""ENERO""","""PRIMER SEMESTRE""","""ENE-MAR""","""MIERCOLES""","""1""","""MONTEVIDEO""","""SECCIONAL 19""","""LA TEJA"""
"""BDF45548""","""VIOLENCIA DOMÉSTICA""","""NO CORRESPONDE""","""NO CORRESPONDE""","""NO""",2013-01-01,2013,"""ENERO""","""PRIMER SEMESTRE""","""ENE-MAR""","""MARTES""","""19""","""MONTEVIDEO""","""SECCIONAL 21""","""COLON CENTRO Y NOROESTE"""


In [6]:
# --- Helper: find column by keyword ---
def find_col(df, keywords):
    """Find the first column whose name contains any of the keywords."""
    for col in df.columns:
        for kw in keywords:
            if kw in col.lower():
                return col
    return None

# Identify key columns
YEAR_COL = find_col(df_delitos, ["anio", "ano", "año", "year"])
DEPT_COL = find_col(df_delitos, ["departamento", "depto", "dept"])
TYPE_COL = find_col(df_delitos, ["titulo", "subtitulo", "tipo_delito", "delito"])

print(f"Year column:       {YEAR_COL}")
print(f"Department column: {DEPT_COL}")
print(f"Type column:       {TYPE_COL}")

# --- Crime type distribution ---
if TYPE_COL:
    crime_dist = (
        df_delitos
        .group_by(TYPE_COL)
        .len()
        .sort("len", descending=True)
        .head(20)
    )
    fig = px.bar(
        crime_dist.to_pandas(),
        x=TYPE_COL, y="len",
        title="Top 20 Crime Types by Report Count",
        labels={TYPE_COL: "Crime Type", "len": "Reports"},
        color="len", color_continuous_scale="Reds",
    )
    fig.update_layout(xaxis_tickangle=-45, height=500)
    fig.show()
else:
    print("No crime type column found")

Year column:       ano
Department column: depto
Type column:       delito


In [7]:
# --- Time series of crime counts by year ---
if YEAR_COL:
    yearly = (
        df_delitos
        .group_by(YEAR_COL)
        .len()
        .sort(YEAR_COL)
    )
    fig = px.line(
        yearly.to_pandas(),
        x=YEAR_COL, y="len",
        title="Reported Crimes Over Time",
        labels={YEAR_COL: "Year", "len": "Total Reports"},
        markers=True,
    )
    fig.update_layout(height=400)
    fig.show()

# --- Department-level heatmap ---
if YEAR_COL and DEPT_COL:
    heatmap_data = (
        df_delitos
        .group_by([DEPT_COL, YEAR_COL])
        .len()
        .sort([DEPT_COL, YEAR_COL])
        .pivot(on=YEAR_COL, index=DEPT_COL, values="len")
        .fill_null(0)
    )
    hm_pd = heatmap_data.to_pandas().set_index(DEPT_COL)
    hm_pd = hm_pd.reindex(sorted(hm_pd.columns, key=str), axis=1)

    fig = px.imshow(
        hm_pd,
        title="Crime Reports by Department and Year",
        labels={"x": "Year", "y": "Department", "color": "Reports"},
        color_continuous_scale="YlOrRd",
        aspect="auto",
    )
    fig.update_layout(height=600)
    fig.show()

In [8]:
# --- Load and profile all other datasets ---
other_datasets = {
    "delitos_homicidios": "delitos_homicidios.parquet",
    "violencia_domestica": "violencia_domestica.parquet",
    "delitos_sexuales": "delitos_sexuales.parquet",
    "homicidios_mujeres": "homicidios_mujeres.parquet",
    "medidas_alternativas": "medidas_alternativas.parquet",
    "sistema_carcelario": "sistema_carcelario.parquet",
}

dfs = {"delitos_denuncias": df_delitos}

for name, filename in other_datasets.items():
    path = DATA_DIR / filename
    df = pl.read_parquet(path)
    dfs[name] = df
    print(f"{'=' * 60}")
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    print()

Dataset: delitos_homicidios
Shape: (4356, 22)
Columns: ['id_victima', 'fecha', 'ano', 'mes', 'trimestre', 'dia_semana', 'hora', 'departamento', 'jurisdiccion', 'lugar', 'motivo_aparente', 'tipo', 'armarec', 'procesados', 'menorescinicioproc', 'aclarado', 'sexo', 'edadcalc', 'nacionalidad', 'antecedentes', 'antecedentesporestupefacientes', 'rel_vict_agres']

First 3 rows:
shape: (3, 22)
┌────────────┬────────────┬──────┬───────┬───┬─────────────┬─────────────┬────────────┬────────────┐
│ id_victima ┆ fecha      ┆ ano  ┆ mes   ┆ … ┆ nacionalida ┆ antecedente ┆ antecedent ┆ rel_vict_a │
│ ---        ┆ ---        ┆ ---  ┆ ---   ┆   ┆ d           ┆ s           ┆ esporestup ┆ gres       │
│ str        ┆ date       ┆ i64  ┆ str   ┆   ┆ ---         ┆ ---         ┆ efacientes ┆ ---        │
│            ┆            ┆      ┆       ┆   ┆ str         ┆ str         ┆ ---        ┆ str        │
│            ┆            ┆      ┆       ┆   ┆             ┆             ┆ str        ┆            │
╞════

In [9]:
# --- Cross-dataset summary using DuckDB ---
con = duckdb.connect()

for name, df in dfs.items():
    con.register(name, df.to_pandas())

# Build UNION query dynamically from all loaded datasets
parts = [f"SELECT '{name}' AS dataset, COUNT(*) AS row_count FROM {name}" for name in dfs]
summary_query = " UNION ALL ".join(parts) + " ORDER BY row_count DESC"

summary_df = con.execute(summary_query).fetchdf()
print("Cross-Dataset Summary:")
print(summary_df.to_string(index=False))

# Per-dataset column count
print("\nColumn counts:")
for name, df in sorted(dfs.items()):
    print(f"  {name}: {len(df.columns)} columns")

fig = px.bar(
    summary_df,
    x="dataset", y="row_count",
    title="Row Counts Across All Tabular Datasets",
    labels={"dataset": "Dataset", "row_count": "Number of Rows"},
    color="row_count", color_continuous_scale="Blues",
)
fig.update_layout(xaxis_tickangle=-30, height=400)
fig.show()

con.close()

Cross-Dataset Summary:
             dataset  row_count
   delitos_denuncias    2423286
 violencia_domestica     532172
    delitos_sexuales      93438
  sistema_carcelario       5012
  delitos_homicidios       4356
  homicidios_mujeres        834
medidas_alternativas        634

Column counts:
  delitos_denuncias: 15 columns
  delitos_homicidios: 22 columns
  delitos_sexuales: 9 columns
  homicidios_mujeres: 15 columns
  medidas_alternativas: 19 columns
  sistema_carcelario: 21 columns
  violencia_domestica: 19 columns


## Findings Summary

### Key Observations

1. **Data Coverage:** Seven datasets spanning different dimensions of public safety
   in Uruguay -- from reported crimes and domestic violence to the prison system.

2. **Column Types:** Most columns are String (categorical) not numeric, which is why
   `describe()` shows null for mean/std/percentiles. The year column (`ano`) is the
   main numeric field. Use `n_unique()` and `value_counts()` for string profiling.

3. **Crime Distribution:** The bar chart reveals which crime types dominate
   reported incidents. Property crimes (hurto, rapina) typically lead.

4. **Temporal Trends:** The time series shows year-over-year patterns in crime
   reporting, highlighting any inflection points or policy impacts.

5. **Geographic Patterns:** The department heatmap reveals spatial concentration
   of crime -- Montevideo and Canelones typically show highest volumes.

6. **Cross-Dataset Scale:** The DuckDB summary provides a quick comparison of
   dataset sizes to understand relative data availability.

### Next Steps

- Join tabular data with geographic layers (see notebook 02)
- Explore the knowledge graph for entity relationships (see notebook 03)
- Build normalized per-capita metrics using population data